In [ ]:
!pip install torch==2.8.0+cu126 --index-url https://download.pytorch.org/whl/cu126
!pip uninstall -y torchvision torchaudio
!pip uninstall -y autoawq awq triton
!pip cache purge
!pip install "autoawq==0.2.9" --no-deps
!pip install "transformers==4.55.4" "accelerate==1.10.1" "sentencepiece==0.2.1" "tokenizers==0.21.4" "huggingface_hub==0.34.4" "safetensors==0.6.2" "numpy==2.0.2" "einops==0.8.1" "tqdm==4.67.1"

> ## **실행 전 해당 런타임에 아래의 .txt 파일이 업로드 되어 있어야 함.**
- chamchiaek.txt
- chamchican.txt
- GreekYogurt.txt
- Latte.txt
- OmletHam.txt

# LGAI-EXAONE-3.5-7.8B AWQ LLM Model 불러오기

In [ ]:
# Model 불러오기
import torch
from transformers import pipeline, AutoTokenizer

MODEL_NAME = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct-AWQ"

pipe = pipeline(
    task="text-generation",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    trust_remote_code=True,
    device_map="auto" if torch.cuda.is_available() else None,
    framework="pt",
)

print("✅ pipeline ready")

# 메시지 형태 테스트
out = pipe([{"role":"user","content":"Who are you?"}],
           max_new_tokens=64, do_sample=False)
print(out[0]["generated_text"])

# 참치캔 시뮬레이션

In [ ]:
import re, csv, os
from typing import List, Tuple

with open("chamchican.txt", "r", encoding="utf-8") as f:
    PERSONAS_RAW = f.read()

print("\n".join(PERSONAS_RAW.splitlines()[:5]))

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple

# =========================
# 0) 전역: 상품/설명
# =========================
DONGWON_VARIANTS = (
    "[동원맛참 참치캔 종류]\n"
    "- 고소 참기름: 고소하고 깊은 맛의 참기름 풍미.\n"
    "- 매콤 참기름: 매콤하고 자극적인 참기름 풍미."
)

MARKET_INFO = (
    "[시장 정보]\n"
    "- 동원맛참 고소 참기름: 135g 4500원 수준.\n"
    "- 동원맛참 고소 참기름: 90g 3200원 수준.\n"
    "- 동원맛참 매콤 참기름: 135g 4500원 수준.\n"
    "- 동원맛참 매콤 참기름: 90g 3200원 수준.\n"
    "- 기타 동원 참치캔 브랜드: 100g 3500원 수준.\n"
)

# =========================
# 1) 페르소나 파서
# =========================
def parse_personas(raw: str) -> List[Tuple[str, str]]:
    """
    PERSONAS_RAW 형식 예:
    Persona_001
    ...설명...
    Persona_002
    ...설명...
    """
    text = raw.strip().replace("\r\n", "\n").replace("\r", "\n")
    header_re = re.compile(r"^\s*\*{0,3}\s*Persona_(\d{3})\s*\*{0,3}\s*:?\s*$", re.M)
    matches = list(header_re.finditer(text))
    pairs: List[Tuple[str, str]] = []
    for i, m in enumerate(matches):
        pid = f"Persona_{m.group(1)}"
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        block = text[start:end].strip()
        if block:
            pairs.append((pid, block))
    pairs.sort(key=lambda x: int(x[0].split("_")[1]))
    return pairs

# 외부에서 PERSONAS_RAW 문자열을 주입해 주세요.
# 예) personas = parse_personas(PERSONAS_RAW)

# =========================
# 2) 프롬프트 템플릿 (숫자 1/2 한 글자 출력 강제)
# =========================
SYSTEM_INST = (
    "너는 시장 조사/소비자 행동 분석 어시스턴트다.\n"
    "규칙:\n"
    " - 아래 [ROLE]의 페르소나 입장에서 참치캔 맛 중 하나를 선택한다.\n"
    " - 출력 형식: 반드시 숫자 '1' 또는 '2' 중 **한 글자만** 출력한다.\n"
    "   1 = 고소 참기름\n"
    "   2 = 매콤 참기름\n"
    " - **숫자 외의 모든 문자, 공백, 개행, 마침표, 설명을 절대 출력하지 않는다.**"
)

def persona_to_role_block(ptext: str) -> str:
    return (
        "[ROLE]\n"
        "아래 페르소나 설명을 그대로 따르는 소비자 역할을 맡아 실제 생활 맥락에서 판단한다.\n"
        f"{ptext}\n"
    )

def build_messages_choose_variant(persona_text: str):
    role_block = persona_to_role_block(persona_text)
    SELECTION_GUIDE = (
    "[선택 기준]\n"
    "- 1(고소참기름): 향이 **은은해야 하는** 가벼운 한 끼나 **아이/매운맛 민감**한 취향에 더 알맞다. 아침·도시락·샐러드·달걀요리·두부처럼 담백한 메뉴, **간장·참깨·마요** 계열과의 조합, 재료 본연의 맛을 **조용히 살리고 싶을 때** 어울린다.\n"
    "- 2(매콤참기름): **김치·마늘·고추장/고춧가루/청양고추** 등과의 한식 상차림에 **일반 성인 입맛** 기준으로 잘 맞고, **느끼함·잡내(생선·기름진 고기·참치마요)를 잡고 식욕을 돋우고 싶을 때** 더 알맞다. **비빔밥·김치볶음밥·비빔면/라면 토핑·덮밥·우동/국수·매콤 볶음·찌개**처럼 존재감을 살릴 메뉴, **야식/안주·캠핑·남은 밥/반찬 활용** 같은 **일상 빈도 높은 맥락**과도 잘 어울린다. 1~2인 가구의 간단한 한 끼에도 매운 향으로 **완성도를 빠르게** 높이기 좋다.\n"
    "편향 금지: 두 선택지 중 어느 한쪽을 기계적으로 선호/회피하지 말고 페르소나의 생활 맥락과 취향에 근거해 판단한다."
    )
    QUESTION = (
        "[질문]\n"
        "참치캔 맛 타입 중에서 무엇을 고를 것인가?\n"
        "선택지는 정확히 다음 둘 중 하나다:\n"
        "  1 = 고소 참기름\n"
        "  2 = 매콤 참기름\n"
        "최종 답변은 반드시 1, 2 중 하나의 숫자만 출력하라."
    )
    user_content = (
        f"{role_block}\n"
        f"{MARKET_INFO}\n\n"
        f"{DONGWON_VARIANTS}\n\n"
        f"{SELECTION_GUIDE}\n\n"
        f"{QUESTION}\n"
    )
    return [
        {"role": "system", "content": SYSTEM_INST},
        {"role": "user",   "content": user_content},
    ]

# =========================
# 3) LLM 호출/검증 (한 글자 산출, 결정론적)
# =========================
# 주의: 아래 pipe는 외부에서 준비된 transformers pipeline(or API) 객체라고 가정합니다.
# 예) pipe = pipeline("chat", model=..., tokenizer=..., device_map="cpu")
GEN_ARGS = dict(
    do_sample=False,
    temperature=0.0,
    top_p=1.0,
    return_full_text=False,
    max_new_tokens=1,   # 한 글자만 생성
)

def _extract_assistant_text_from_pipe_output(out) -> str:
    if not out:
        return ""
    item = out[0]
    if isinstance(item, dict):
        if isinstance(item.get("generated_text"), str):
            return item["generated_text"]
        gen = item.get("generated_text")
        if isinstance(gen, list):
            for m in gen:
                if isinstance(m, dict) and m.get("role") == "assistant":
                    c = m.get("content", "")
                    if isinstance(c, str):
                        return c
        if isinstance(item.get("content"), str):
            return item["content"]
    if isinstance(item, str):
        return item
    return str(item)

def generate_once(messages, **gen_args) -> str:
    out = pipe(messages, **gen_args)
    text = _extract_assistant_text_from_pipe_output(out)
    return text.strip() if text else ""

def parse_variant_num_strict(text: str) -> int:
    """
    보정 없이 엄격 검증: 반드시 '1'/'2' 중 하나여야 함.
    아니면 예외 발생.
    """
    t = (text or "").strip()
    if t not in ("1", "2"):
        raise ValueError(f"모델 출력이 1/2가 아님: {repr(t)}")
    return int(t)

def num_to_label(n: int) -> str:
    return {1: "고소 참기름", 2: "매콤 참기름"}[n]

# =========================
# 4) 실행 & 저장
# =========================
def main(personas: List[Tuple[str, str]], out_path: str = "dongwon_variant_choice_chamchican.csv"):
    if not personas:
        raise ValueError("파싱된 페르소나가 없습니다. PERSONAS_RAW를 확인하세요.")

    rows = []
    total = len(personas)
    for i, (pid, ptext) in enumerate(personas, start=1):
        print(f">>> Persona {i}/{total} - {pid}", flush=True)
        msgs = build_messages_choose_variant(ptext)
        raw = generate_once(msgs, **GEN_ARGS)
        variant_num = parse_variant_num_strict(raw)  # 보정 없이 검증
        choice_label = num_to_label(variant_num)
        rows.append({
            "persona_idx": i,
            "persona_id": pid,
            "variant_num": variant_num,      # 1/2
            "variant_choice": choice_label,  # 고소 참기름 / 매콤 참기름
            "raw_reply": raw,                # 원문(숫자 1글자)
        })

    with open(out_path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(
            f,
            fieldnames=["persona_idx","persona_id","variant_num","variant_choice","raw_reply"]
        )
        w.writeheader()
        w.writerows(rows)

    print(f"\n✅ Saved: {os.path.abspath(out_path)} (rows={len(rows)})")

# 사용 예시:
personas = parse_personas(PERSONAS_RAW)
if __name__ == "__main__":
    main(personas)


In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd

IN_FILE = "dongwon_variant_choice_chamchican.csv"
OUT_SUMMARY = "dongwon_variant_choice_chamchican_summary.csv"

NUM2LABEL = {1: "고소 참기름", 2: "매콤 참기름"}

def get_choice_stats(in_path: str = IN_FILE):
    """
    CSV를 읽어 1/2 선택의 개수, 퍼센트(0~100), ratio(합=1)를 계산해 반환.
    반환값: (stats_dict, df, summary_df)
      - stats_dict: {'total': int, 1: {...}, 2: {...}}
    """
    # 1) 로드 & 컬럼 확인
    df = pd.read_csv(in_path, encoding="utf-8-sig")
    required_cols = {"persona_idx","persona_id","variant_num","variant_choice","raw_reply"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"CSV에 필요한 컬럼이 없습니다: {missing}")

    # 2) 숫자 정합성 보정 (혹시 모를 비정상 값 방지)
    df["variant_num"] = pd.to_numeric(df["variant_num"], errors="coerce").fillna(2).astype(int)
    df["variant_num"] = df["variant_num"].where(df["variant_num"].isin([1, 2]), 2)
    df["variant_choice"] = df["variant_num"].map(NUM2LABEL)

    # 3) 집계
    order = [1, 2]
    counts = df["variant_num"].value_counts().reindex(order, fill_value=0)
    total = int(counts.sum())
    ratios = (counts / total).round(6)              # 합=1
    percents = (ratios * 100).round(2)              # 0~100

    # 4) 요약 테이블(DataFrame)
    summary_df = pd.DataFrame({
        "variant_num": order,
        "variant_choice": [NUM2LABEL[n] for n in order],
        "count": counts.values.astype(int),
        "percent": percents.values,                 # 0~100
        "ratio": ratios.values                      # 합=1
    })

    # 5) stats 딕셔너리(반환용)
    stats = {
        "total": total,
        1: {"label": NUM2LABEL[1], "count": int(counts[1]), "percent": float(percents[1]), "ratio": float(ratios[1])},
        2: {"label": NUM2LABEL[2], "count": int(counts[2]), "percent": float(percents[2]), "ratio": float(ratios[2])},
    }

    return stats, df, summary_df

def analyze_and_save(in_path: str = IN_FILE, out_path: str = OUT_SUMMARY):
    stats, df, summary = get_choice_stats(in_path)

    # 콘솔 출력
    print(f"총 응답 수: {stats['total']}명")
    for n in [1, 2]:
        s = stats[n]
        print(f"{n} = {s['label']}: {s['count']}명 ({s['percent']}%), ratio={s['ratio']:.3f}")

    print("\n=== 요약 테이블 ===")
    print(summary.to_string(index=False))

    # 저장
    summary.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ 요약 저장: {out_path}")

    return stats, summary  # 필요한 곳에서 바로 활용 가능

if __name__ == "__main__":
    _, _ = analyze_and_save()


# 고소 참기름 시뮬레이션

In [ ]:
# =========================
# 고소참기름 구매자: 용량(135g vs 90g) 선택 & 저장
# =========================

# (1) 사이즈 선택용 시스템 프롬프트
TUNA_SIZE_SYSTEM_INST = (
    "너는 시장 조사/소비자 행동 분석 어시스턴트다.\n"
    "규칙:\n"
    " - 아래 [ROLE]의 페르소나 입장에서 '참치(고소참기름)' 구매 시 용량 하나를 선택한다.\n"
    " - 출력 형식: 반드시 숫자 '1' 또는 '2' 중 **한 글자만** 출력한다.\n"
    "   1 = 135g\n"
    "   2 = 90g\n"
    " - **숫자 외의 모든 문자, 공백, 개행, 마침표, 설명을 절대 출력하지 않는다.**"
)

def build_messages_choose_tuna_size(persona_text: str):
    role_block = persona_to_role_block(persona_text)
    TUNA_SIZE_SELECTION_GUIDE = (
    "[선택 기준]\n"
  "- 1(135g): 덮밥·볶음·비빔밥처럼 한 끼에 넉넉히 소비하는 메뉴가 주로 나오며, 가성비·양을 중시하고 남은 양을 다음날 이어 쓰는 습관이 뚜렷하다면 135g이 알맞다. 주로 식사량이 많은 청년 남성층에게 적합하다.\n"
  "- 2(90g): 참치를 주 1–2회 정도 드문드문 사용하거나, 샐러드·도시락·토핑·간단 반찬 등 가볍게 활용하는 경우에 적합하다. 또한 자주 요리하더라도 한 번에 쓰는 양이 많지 않거나, 신선도·칼로리·간편성을 중시한다면 90g이 더 알맞다. 특히 2인 가구라도 소량 위주 소비 패턴이면 90g이 적합하며,일반적인 한끼 식사에 최적이다.\n"
    "편향 금지: 두 선택지 중 어느 한쪽을 기계적으로 선호/회피하지 말고 페르소나의 생활 맥락과 취향에 근거해 판단한다."
    )
    TUNA_QUESTION_SIZE = (
        "[질문]\n"
        "참치(고소참기름)를 구매할 때 어떤 용량을 고를 것인가?\n"
        "선택지는 정확히 다음 둘 중 하나다:\n"
        "  1 = 135g\n"
        "  2 = 90g\n"
        "최종 답변은 반드시 1, 2 중 하나의 숫자만 출력하라."
    )
    user_content = (
        f"{role_block}\n"
        f"{MARKET_INFO}\n\n"        # 현재 스코프의 참치 가격/시장정보 사용
        f"{DONGWON_VARIANTS}\n\n"   # 현재 스코프의 참치 제품 설명 사용
        f"{TUNA_SIZE_SELECTION_GUIDE}\n\n"
        f"{TUNA_QUESTION_SIZE}\n"
    )
    return [
        {"role": "system", "content": TUNA_SIZE_SYSTEM_INST},
        {"role": "user",   "content": user_content},
    ]

# (2) '1/2' 엄격 파서 (사이즈용)
def parse_tuna_size_num_strict(text: str) -> int:
    t = (text or "").strip()
    if t not in ("1", "2"):
        raise ValueError(f"모델 출력이 1/2가 아님: {repr(t)}")
    return int(t)

def tuna_size_num_to_label(n: int) -> str:
    return {1: "135g", 2: "90g"}[n]

# (3) 실행 함수: 이전 CSV에서 '고소참기름' 구매자만 추출해 용량 선택 저장
def run_size_choice_for_goso_tuna_buyers(
    prev_csv_path: str = "dongwon_variant_choice_tuna.csv",
    out_path: str = "tuna_goso_size_choice.csv"
):
    # 페르소나 맵 (id -> 텍스트)
    personas_list = parse_personas(PERSONAS_RAW)
    pid_to_text = {pid: ptext for pid, ptext in personas_list}

    targets = []
    with open(prev_csv_path, encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            # '고소참기름(1)'만 대상 (필요 시 variant_choice 문자열도 함께 체크)
            if row.get("variant_num") == "1" or row.get("variant_choice") == "고소참기름":
                targets.append(row)

    if not targets:
        print("대상(고소참기름 선택자)이 없습니다.")
        return

    results = []
    total = len(targets)
    for i, row in enumerate(targets, start=1):
        pid = row.get("persona_id")
        ptext = pid_to_text.get(pid, "")
        if not ptext:
            print(f"경고: 페르소나 텍스트 누락 - {pid}, 건너뜀")
            continue

        print(f">>> Tuna Size {i}/{total} - {pid}", flush=True)
        msgs = build_messages_choose_tuna_size(ptext)
        raw = generate_once(msgs, **GEN_ARGS)
        size_num = parse_tuna_size_num_strict(raw)
        size_label = tuna_size_num_to_label(size_num)

        results.append({
            "persona_id": pid,
            "persona_idx": row.get("persona_idx"),
            "variant_num_prev": row.get("variant_num"),
            "variant_choice_prev": row.get("variant_choice"),
            "size_num": size_num,          # 1/2
            "size_choice": size_label,     # 135g / 90g
            "raw_reply": raw,              # 원문(숫자 1글자)
        })

    # 새 CSV로 저장 (고소참기름 선택자만)
    with open(out_path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(
            f,
            fieldnames=[
                "persona_id","persona_idx",
                "variant_num_prev","variant_choice_prev",
                "size_num","size_choice","raw_reply"
            ]
        )
        w.writeheader()
        w.writerows(results)

    print(f"\n✅ Saved tuna size choices: {os.path.abspath(out_path)} (rows={len(results)})")

# 사용 예시:
run_size_choice_for_goso_tuna_buyers(
    prev_csv_path="/content/dongwon_variant_choice_chamchican.csv",
    out_path="/content/tuna_goso_size_choice.csv"
    )


In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd

IN_FILE = "tuna_goso_size_choice.csv"
OUT_SUMMARY = "tuna_goso_size_choice_summary.csv"

NUM2LABEL = {1: "135g", 2: "90g"}

def analyze_tuna_size_choices(in_path: str = IN_FILE, out_path: str = OUT_SUMMARY):
    # 1) 데이터 로드
    df = pd.read_csv(in_path, encoding="utf-8-sig")

    # 2) 최소 필요 컬럼 확인
    required_cols = {"persona_id","persona_idx","size_num","size_choice","raw_reply"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"CSV에 필요한 컬럼이 없습니다: {missing}")

    # 3) 숫자/라벨 정합성 보정 (비정상 값 방지)
    df["size_num"] = pd.to_numeric(df["size_num"], errors="coerce").fillna(2).astype(int)
    df["size_num"] = df["size_num"].where(df["size_num"].isin([1, 2]), 2)
    df["size_choice"] = df["size_num"].map(NUM2LABEL)

    # 4) 집계
    order = [1, 2]
    counts = df["size_num"].value_counts().reindex(order, fill_value=0).astype(int)
    total = int(counts.sum())
    ratios = (counts / total).round(6)          # 0~1, 합=1
    percents = (ratios * 100).round(2)          # 0~100 %

    # 5) 요약 테이블
    summary = pd.DataFrame({
        "size_num": order,
        "size_choice": [NUM2LABEL[n] for n in order],
        "count": counts.values,
        "percent": percents.values,             # 0~100
        "ratio": ratios.values                  # 합=1
    })

    # 6) 출력
    print("=== 고소참기름 구매자: 용량 선택 요약 ===")
    print(f"총 대상 수: {total}명")
    for n in order:
        print(f"{n} = {NUM2LABEL[n]}: {counts[n]}명 ({percents[n]}%), ratio={ratios[n]:.3f}")

    print("\n=== 요약 테이블 ===")
    print(summary.to_string(index=False))

    # 7) 저장
    summary.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ 요약 저장: {out_path}")

    # 8) 반환(추가 분석용)
    return df, summary

if __name__ == "__main__":
    _, _ = analyze_tuna_size_choices()


# 매콤 참기름 시뮬레이션

In [ ]:
# =========================
# 매콤참기름 구매자: 용량(135g vs 90g) 선택 & 저장
# =========================

# (1) 사이즈 선택용 시스템 프롬프트
SPICY_TUNA_SIZE_SYSTEM_INST = (
    "너는 시장 조사/소비자 행동 분석 어시스턴트다.\n"
    "규칙:\n"
    " - 아래 [ROLE]의 페르소나 입장에서 '참치(매콤참기름)' 구매 시 용량 하나를 선택한다.\n"
    " - 출력 형식: 반드시 숫자 '1' 또는 '2' 중 **한 글자만** 출력한다.\n"
    "   1 = 135g\n"
    "   2 = 90g\n"
    " - **숫자 외의 모든 문자, 공백, 개행, 마침표, 설명을 절대 출력하지 않는다.**"
)

def build_messages_choose_spicy_tuna_size(persona_text: str):
    role_block = persona_to_role_block(persona_text)
    TUNA_SIZE_SELECTION_GUIDE = (
    "[선택 기준]\n"
    "- 1(135g): 2인 이상이 함께 먹는 끼니가 잦거나, 1~2인이라도 주 2~3회 이상 사용하면서 비빔·볶음·덮밥·찌개 등 조리형 메뉴에서 한 끼 사용량이 보통 이상(≈1.5~2인분)일 때 적합하다. 반 캔을 다음 끼니로 이어 쓰는 계획이 있거나 소분·냉장/냉동으로 2~3일 내 연속 소진이 가능하고, 그람당 단가(가성비)도 어느 정도 고려한다면 1이 자연스럽다.\n"
    "- 2(90g): 1인 한 끼를 남김 없이 쓰려 하거나 주 0~2회 사용 + 토핑/샐러드·샌드위치 등 소량 위주일 때 적합하다. 다음 끼니로 이어 쓰지 않음, 메뉴를 자주 바꾸는 취향, 신선도·칼로리/염분 관리, 이동·도시락/오피스, 냉장 공간 협소·공용 냉장고 환경이라면 2가 편하다.\n"
    "편향 금지: 두 선택지 중 어느 한쪽을 기계적으로 선호/회피하지 말고 페르소나의 생활 맥락과 취향에 근거해 판단한다."
    )
    TUNA_QUESTION_SIZE = (
        "[질문]\n"
        "참치(매콤참기름)를 구매할 때 어떤 용량을 고를 것인가?\n"
        "선택지는 정확히 다음 둘 중 하나다:\n"
        "  1 = 135g\n"
        "  2 = 90g\n"
        "최종 답변은 반드시 1, 2 중 하나의 숫자만 출력하라."
    )

    user_content = (
        f"{role_block}\n"
        f"{MARKET_INFO}\n\n"
        f"{DONGWON_VARIANTS}\n\n"
        f"{TUNA_SIZE_SELECTION_GUIDE}\n\n"
        f"{TUNA_QUESTION_SIZE}\n"
    )
    return [
        {"role": "system", "content": SPICY_TUNA_SIZE_SYSTEM_INST},
        {"role": "user",   "content": user_content},
    ]

# (2) '1/2' 엄격 파서 (사이즈용)
def parse_spicy_tuna_size_num_strict(text: str) -> int:
    t = (text or "").strip()
    if t not in ("1", "2"):
        raise ValueError(f"모델 출력이 1/2가 아님: {repr(t)}")
    return int(t)

def spicy_tuna_size_num_to_label(n: int) -> str:
    return {1: "135g", 2: "90g"}[n]

# (3) 실행 함수: 이전 CSV에서 '매콤참기름' 구매자만 추출해 용량 선택 저장
def run_size_choice_for_spicy_tuna_buyers(
    prev_csv_path: str = "dongwon_variant_choice_tuna.csv",
    out_path: str = "tuna_spicy_size_choice.csv"
):
    # 페르소나 맵 (id -> 텍스트)
    personas_list = parse_personas(PERSONAS_RAW)
    pid_to_text = {pid: ptext for pid, ptext in personas_list}

    targets = []
    with open(prev_csv_path, encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            # '매콤참기름(2)'만 대상
            if row.get("variant_num") == "2" or row.get("variant_choice") == "매콤참기름":
                targets.append(row)

    if not targets:
        print("대상(매콤참기름 선택자)이 없습니다.")
        return

    results = []
    total = len(targets)
    for i, row in enumerate(targets, start=1):
        pid = row.get("persona_id")
        ptext = pid_to_text.get(pid, "")
        if not ptext:
            print(f"경고: 페르소나 텍스트 누락 - {pid}, 건너뜀")
            continue

        print(f">>> Spicy Tuna Size {i}/{total} - {pid}", flush=True)
        msgs = build_messages_choose_spicy_tuna_size(ptext)
        raw = generate_once(msgs, **GEN_ARGS)
        size_num = parse_spicy_tuna_size_num_strict(raw)
        size_label = spicy_tuna_size_num_to_label(size_num)

        results.append({
            "persona_id": pid,
            "persona_idx": row.get("persona_idx"),
            "variant_num_prev": row.get("variant_num"),
            "variant_choice_prev": row.get("variant_choice"),
            "size_num": size_num,          # 1/2
            "size_choice": size_label,     # 135g / 90g
            "raw_reply": raw,              # 원문(숫자 1글자)
        })

    # 새 CSV로 저장 (매콤참기름 선택자만)
    with open(out_path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(
            f,
            fieldnames=[
                "persona_id","persona_idx",
                "variant_num_prev","variant_choice_prev",
                "size_num","size_choice","raw_reply"
            ]
        )
        w.writeheader()
        w.writerows(results)

    print(f"\n✅ Saved spicy tuna size choices: {os.path.abspath(out_path)} (rows={len(results)})")

# 사용 예시:
run_size_choice_for_spicy_tuna_buyers(
    prev_csv_path="/content/dongwon_variant_choice_chamchican.csv",
    out_path="/content/tuna_spicy_size_choice.csv"
    )


In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd

IN_FILE = "tuna_spicy_size_choice.csv"
OUT_SUMMARY = "tuna_spicy_size_choice_summary.csv"

NUM2LABEL = {1: "135g", 2: "90g"}

def analyze_spicy_tuna_size_choices(in_path: str = IN_FILE, out_path: str = OUT_SUMMARY):
    # 1) 데이터 로드
    df = pd.read_csv(in_path, encoding="utf-8-sig")

    # 2) 최소 필요 컬럼 확인
    required_cols = {"persona_id","persona_idx","size_num","size_choice","raw_reply"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"CSV에 필요한 컬럼이 없습니다: {missing}")

    # 3) 숫자/라벨 정합성 보정
    df["size_num"] = pd.to_numeric(df["size_num"], errors="coerce").fillna(2).astype(int)
    df["size_num"] = df["size_num"].where(df["size_num"].isin([1, 2]), 2)
    df["size_choice"] = df["size_num"].map(NUM2LABEL)

    # 4) 집계
    order = [1, 2]
    counts = df["size_num"].value_counts().reindex(order, fill_value=0).astype(int)
    total = int(counts.sum())

    if total > 0:
        ratios = (counts / total).round(6)       # 합=1
        percents = (ratios * 100).round(2)       # 0~100 %
    else:
        # 빈 파일/행만 있는 경우 방어적 처리
        ratios = pd.Series([0.0, 0.0], index=order)
        percents = pd.Series([0.0, 0.0], index=order)

    # 5) 요약 테이블
    summary = pd.DataFrame({
        "size_num": order,
        "size_choice": [NUM2LABEL[n] for n in order],
        "count": counts.values,
        "percent": percents.values,              # 0~100
        "ratio": ratios.values                   # 합=1
    })

    # 6) 출력
    print("=== 매콤참기름 구매자: 용량 선택 요약 ===")
    print(f"총 대상 수: {total}명")
    for n in order:
        print(f"{n} = {NUM2LABEL[n]}: {counts[n]}명 ({percents[n]}%), ratio={ratios[n]:.3f}")

    print("\n=== 요약 테이블 ===")
    print(summary.to_string(index=False))

    # 7) 저장
    summary.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ 요약 저장: {out_path}")

    # 8) 반환(추가 분석용)
    return df, summary

if __name__ == "__main__":
    _, _ = analyze_spicy_tuna_size_choices()


# 참치캔 시뮬레이션 최종 정리

고소 : 매콤 = 66 : 34 = 1.9 : 1

고소 → 135g : 90g = 32 : 34 = 0.9 : 1

매콤  → 135g : 90g = 11 : 23 = 0.5 : 1

# 통조림 햄 시뮬레이션

In [ ]:
import re, csv, os
from typing import List, Tuple

with open("OmeletHam.txt", "r", encoding="utf-8") as f:
    PERSONAS_RAW = f.read()

print("\n".join(PERSONAS_RAW.splitlines()[:5]))

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple

# =========================
# 0) 전역: 상품/설명 (참고용, 균형 서술)
# =========================
DONGWON_VARIANTS = (
    "[동원 신제품 종류]\n"
    "- 오믈레햄: 계란·케첩 풍미의 순한 단짠."
)

MARKET_INFO = (
    "[시장 정보]\n"
    "- 동원 통조림 햄(리챔 오리지널): 200g 6200원 수준.\n"
    "- 동원 통조림 햄(리챔 오리지널): 340g 8200원 수준.\n"
    "- 동원 통조림 햄(리챔 오믈레햄): 200g 5080원 수준.\n"
    "- 동원 통조림 햄(리챔 오믈레햄): 340g 7480원 수준.\n"
    "- CJ 통조림 햄(스팸 클래식): 200g 6900원 수준.\n"
    "- CJ 통조림 햄(스팸 클래식): 340g 8000원 수준.\n"
    "- 롯데 통조림 햄(로스팜): 200g 6200원 수준.\n"
    "- 롯데 통조림 햄(로스팜): 340g 8200원 수준.\n"
)

# =========================
# 1) 페르소나 파서
# =========================
def parse_personas(raw: str) -> List[Tuple[str, str]]:
    """
    PERSONAS_RAW 형식 예:
    Persona_001
    ...설명...
    Persona_002
    ...설명...
    """
    text = raw.strip().replace("\r\n", "\n").replace("\r", "\n")
    header_re = re.compile(r"^\s*\*{0,3}\s*Persona_(\d{3})\s*\*{0,3}\s*:?\s*$", re.M)
    matches = list(header_re.finditer(text))
    pairs: List[Tuple[str, str]] = []
    for i, m in enumerate(matches):
        pid = f"Persona_{m.group(1)}"
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        block = text[start:end].strip()
        if block:
            pairs.append((pid, block))
    pairs.sort(key=lambda x: int(x[0].split("_")[1]))
    return pairs

# 외부에서 PERSONAS_RAW 문자열을 주입해 주세요.
# 예) personas = parse_personas(PERSONAS_RAW)

# =========================
# 2) 프롬프트 템플릿 (숫자 1/2 한 글자 출력 강제)
# =========================
SYSTEM_INST = (
    "너는 시장 조사/소비자 행동 분석 어시스턴트다.\n"
    "규칙:\n"
    " - 아래 [ROLE]의 페르소나 입장에서 통조림 햄 맛 중 하나를 선택한다.\n"
    " - 출력 형식: 반드시 숫자 '1' 또는 '2' 중 **한 글자만** 출력한다.\n"
    "   1 = 리챔 오리지널\n"
    "   2 = 리챔 오믈레햄\n"
    " - **숫자 외의 모든 문자, 공백, 개행, 마침표, 설명을 절대 출력하지 않는다.**"
)

def persona_to_role_block(ptext: str) -> str:
    return (
        "[ROLE]\n"
        "아래 페르소나 설명을 그대로 따르는 소비자 역할을 맡아 실제 생활 맥락에서 판단한다.\n"
        f"{ptext}\n"
    )

def build_messages_choose_variant(persona_text: str):
    role_block = persona_to_role_block(persona_text)
    SELECTION_GUIDE = (
    "[선택 기준]\n"
    "- 1(리챔 오리지널): 담백·짭짤한 기본 햄 풍미와 탄탄한 식감을 선호하는 경향에 특히 적합하다. 간·수분·기름과 굽기 정도를 조절해 **맛/식감 변주가 쉽고 자유도**가 높다. 국물(찌개/부대찌개)·각종 볶음(김치/야채/밥)·김밥/주먹밥·라면/국수 토핑·팬구이 반찬·안주·샌드위치/토스트·에어프라이어 등 **폭넓은 조리법과 다양한 양념(간장·고추장·마늘·후추·허브)**과의 **호환성**이 높아, 가정식·야식·캠핑·손님 상 등 **여러 일상 상황**과도 잘 어울린다. 매콤/짭짤 계열을 즐기거나, **재료 본연의 맛을 살리면서 조미를 직접 맞추길 선호**하는 취향에도 유리하다.\n"
    "- 2(리챔 오믈레햄): 계란과 토마토계 소스(케첩)의 **순한 단짠 콘셉트 자체**를 단품으로 간단히 활용하려는 **아주 제한적인** 맥락에 더 알맞다. **오므라이스 1인분 또는 간단 도시락 1인분**을 **한 팬·짧은 시간**에 끝내려는 경우, **별도 계란/소스 추가 계획이 없는** 간단 조리, **아이 중심의 온순한 맛**을 원하는 상황과 어울린다. 국물·매콤 볶음·김밥류·라면 토핑·바삭한 팬구이처럼 **양념을 달리하며 폭넓게 응용**하려는 조리와는 방향이 다를 수 있다.\n"
    "편향 금지: 두 선택지 중 어느 한쪽을 기계적으로 선호/회피하지 말고 페르소나의 생활 맥락과 취향에 근거해 판단한다."
    )
    QUESTION = (
        "[질문]\n"
        "통조림 햄 맛 타입 중에서 무엇을 고를 것인가?\n"
        "선택지는 정확히 다음 둘 중 하나다:\n"
        "  1 = 리챔 오리지널\n"
        "  2 = 리챔 오믈레햄\n"
        "최종 답변은 반드시 1, 2 중 하나의 숫자만 출력하라."
    )
    user_content = (
        f"{role_block}\n"
        f"{MARKET_INFO}\n\n"
        f"{DONGWON_VARIANTS}\n\n"
        f"{SELECTION_GUIDE}\n\n"
        f"{QUESTION}\n"
    )
    return [
        {"role": "system", "content": SYSTEM_INST},
        {"role": "user",   "content": user_content},
    ]

# =========================
# 3) LLM 호출/검증 (한 글자 산출, 결정론적)
# =========================
# 주의: 아래 pipe는 외부에서 준비된 transformers pipeline(or API) 객체라고 가정합니다.
# 예) pipe = pipeline("chat", model=..., tokenizer=..., device_map="cpu")
GEN_ARGS = dict(
    do_sample=False,
    temperature=0.0,
    top_p=1.0,
    return_full_text=False,
    max_new_tokens=1,   # 한 글자만 생성
)

def _extract_assistant_text_from_pipe_output(out) -> str:
    if not out:
        return ""
    item = out[0]
    if isinstance(item, dict):
        if isinstance(item.get("generated_text"), str):
            return item["generated_text"]
        gen = item.get("generated_text")
        if isinstance(gen, list):
            for m in gen:
                if isinstance(m, dict) and m.get("role") == "assistant":
                    c = m.get("content", "")
                    if isinstance(c, str):
                        return c
        if isinstance(item.get("content"), str):
            return item["content"]
    if isinstance(item, str):
        return item
    return str(item)

def generate_once(messages, **gen_args) -> str:
    out = pipe(messages, **gen_args)
    text = _extract_assistant_text_from_pipe_output(out)
    return text.strip() if text else ""

def parse_variant_num_strict(text: str) -> int:
    """
    보정 없이 엄격 검증: 반드시 '1'/'2' 중 하나여야 함.
    아니면 예외 발생.
    """
    t = (text or "").strip()
    if t not in ("1", "2"):
        raise ValueError(f"모델 출력이 1/2가 아님: {repr(t)}")
    return int(t)

def num_to_label(n: int) -> str:
    return {1: "리챔 오리지널", 2: "리챔 오믈레햄"}[n]

# =========================
# 4) 실행 & 저장
# =========================
def main(personas: List[Tuple[str, str]], out_path: str = "dongwon_variant_choice_omeletham.csv"):
    if not personas:
        raise ValueError("파싱된 페르소나가 없습니다. PERSONAS_RAW를 확인하세요.")

    rows = []
    total = len(personas)
    for i, (pid, ptext) in enumerate(personas, start=1):
        print(f">>> Persona {i}/{total} - {pid}", flush=True)
        msgs = build_messages_choose_variant(ptext)
        raw = generate_once(msgs, **GEN_ARGS)
        variant_num = parse_variant_num_strict(raw)  # 보정 없이 검증
        choice_label = num_to_label(variant_num)
        rows.append({
            "persona_idx": i,
            "persona_id": pid,
            "variant_num": variant_num,      # 1/2
            "variant_choice": choice_label,  # 리챔 오리지널 / 리챔 오믈레햄
            "raw_reply": raw,                # 원문(숫자 1글자)
        })

    with open(out_path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(
            f,
            fieldnames=["persona_idx","persona_id","variant_num","variant_choice","raw_reply"]
        )
        w.writeheader()
        w.writerows(rows)

    print(f"\n✅ Saved: {os.path.abspath(out_path)} (rows={len(rows)})")

# 사용 예시:
personas = parse_personas(PERSONAS_RAW)
if __name__ == "__main__":
    main(personas)


In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd

IN_FILE = "dongwon_variant_choice_omeletham.csv"
OUT_SUMMARY = "dongwon_variant_choice_omeletham_summary.csv"

NUM2LABEL = {1: "리챔 오리지널", 2: "리챔 오믈레햄"}

def analyze_omeletham_variant_choices(in_path: str = IN_FILE, out_path: str = OUT_SUMMARY):
    # 1) 데이터 로드
    df = pd.read_csv(in_path, encoding="utf-8-sig")

    # 2) 최소 필요 컬럼 확인
    required_cols = {"persona_idx","persona_id","variant_num","variant_choice","raw_reply"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"CSV에 필요한 컬럼이 없습니다: {missing}")

    # 3) 숫자/라벨 정합성 보정 (숫자 기준으로 라벨 재매핑)
    df["variant_num"] = pd.to_numeric(df["variant_num"], errors="coerce").fillna(2).astype(int)
    df["variant_num"] = df["variant_num"].where(df["variant_num"].isin([1, 2]), 2)
    df["variant_choice"] = df["variant_num"].map(NUM2LABEL)

    # 4) 집계
    order = [1, 2]
    counts = df["variant_num"].value_counts().reindex(order, fill_value=0).astype(int)
    total = int(counts.sum())

    if total > 0:
        ratios = (counts / total).round(6)    # 0~1, 합=1
        percents = (ratios * 100).round(2)    # 0~100 %
    else:
        # 빈 데이터 방어 처리
        ratios = pd.Series([0.0, 0.0], index=order)
        percents = pd.Series([0.0, 0.0], index=order)

    # 5) 요약 테이블
    summary = pd.DataFrame({
        "variant_num": order,
        "variant_choice": [NUM2LABEL[n] for n in order],
        "count": counts.values,
        "percent": percents.values,   # 0~100
        "ratio": ratios.values        # 합=1
    })

    # 6) 출력
    print("=== 오믈레햄 실험: 맛 선택 요약 ===")
    print(f"총 응답 수: {total}명")
    for n in order:
        print(f"{n} = {NUM2LABEL[n]}: {counts[n]}명 ({percents[n]}%), ratio={ratios[n]:.3f}")

    print("\n=== 요약 테이블 ===")
    print(summary.to_string(index=False))

    # 7) 저장
    summary.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ 요약 저장: {out_path}")

    # 8) 반환(추가 분석용)
    return df, summary

if __name__ == "__main__":
    _, _ = analyze_omeletham_variant_choices()


# 오믈레햄 시뮬레이션

In [ ]:
# =========================
# 5) 오믈레햄 구매자: 용량(200g vs 340g) 선택 & 저장
# =========================

# (1) 사이즈 선택용 시스템 프롬프트
SIZE_SYSTEM_INST = (
    "너는 시장 조사/소비자 행동 분석 어시스턴트다.\n"
    "규칙:\n"
    " - 아래 [ROLE]의 페르소나 입장에서 '리챔 오믈레햄' 구매 시 용량 하나를 선택한다.\n"
    " - 출력 형식: 반드시 숫자 '1' 또는 '2' 중 **한 글자만** 출력한다.\n"
    "   1 = 200g\n"
    "   2 = 340g\n"
    " - **숫자 외의 모든 문자, 공백, 개행, 마침표, 설명을 절대 출력하지 않는다.**"
)

def build_messages_choose_size(persona_text: str):
    role_block = persona_to_role_block(persona_text)
    SIZE_SELECTION_GUIDE = (
    "[선택 기준]\n"
    "- 1(200g): 1인 가구이거나 **2인 가구**에서 온라인몰 중심으로 구매하며, 개봉 후 **남김/보관 부담을 줄이고** 소량씩 **자주** 쓰려는 패턴에 더 알맞다. 도시락/한 끼 분량, 다양한 메뉴를 **나눠 쓰기**, 냉장 공간이 협소한 경우와 어울린다.\n"
    "- 2(340g): 다음 조건에 **대체로 모두 해당**할 때 더 알맞다: **오프라인 대형마트 중심 구매**, **찌개·부대찌개 같은 한 냄비 메뉴를 주 3회 이상** 조리, **소분·냉동 보관 계획(냉동실 여유)**. 또는 **3인 이상이 같은 찌개 메뉴를 함께** 먹는 경우. 그 외에는 200g이 관리·회전 측면에서 더 무난할 수 있다.\n"
    "편향 금지: 두 선택지 중 어느 한쪽을 기계적으로 선호/회피하지 말고 페르소나의 생활 맥락과 취향에 근거해 판단한다."
    )
    QUESTION_SIZE = (
        "[질문]\n"
        "리챔 오믈레햄을 구매할 때 어떤 용량을 고를 것인가?\n"
        "선택지는 정확히 다음 둘 중 하나다:\n"
        "  1 = 200g\n"
        "  2 = 340g\n"
        "최종 답변은 반드시 1, 2 중 하나의 숫자만 출력하라."
    )
    user_content = (
        f"{role_block}\n"
        f"{MARKET_INFO}\n\n"         # 기존 가격 정보 활용
        f"{DONGWON_VARIANTS}\n\n"    # 오믈레햄 간단 설명
        f"{SIZE_SELECTION_GUIDE}\n\n"
        f"{QUESTION_SIZE}\n"
    )
    return [
        {"role": "system", "content": SIZE_SYSTEM_INST},
        {"role": "user",   "content": user_content},
    ]

# (2) '1/2' 엄격 파서 (사이즈용)
def parse_size_num_strict(text: str) -> int:
    t = (text or "").strip()
    if t not in ("1", "2"):
        raise ValueError(f"모델 출력이 1/2가 아님: {repr(t)}")
    return int(t)

def size_num_to_label(n: int) -> str:
    return {1: "200g", 2: "340g"}[n]

# (3) 실행 함수: 이전 CSV에서 오믈레햄 구매자만 추출해 용량 선택 저장
def run_size_choice_for_omeletham_buyers(
    prev_csv_path: str = "dongwon_variant_choice_omeletham18.csv",
    out_path: str = "dongwon_omeletham_size_choice.csv"
):
    # 페르소나 맵 다시 구성 (id -> text)
    personas_list = parse_personas(PERSONAS_RAW)
    pid_to_text = {pid: ptext for pid, ptext in personas_list}

    targets = []
    with open(prev_csv_path, encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            # 기존 결과에서 '오믈레햄(2)'만 대상
            if row.get("variant_num") == "2" or row.get("variant_choice") == "리챔 오믈레햄":
                targets.append(row)

    if not targets:
        print("대상(오믈레햄 선택자)이 없습니다.")
        return

    results = []
    total = len(targets)
    for i, row in enumerate(targets, start=1):
        pid = row.get("persona_id")
        ptext = pid_to_text.get(pid, "")
        if not ptext:
            print(f"경고: 페르소나 텍스트 누락 - {pid}, 건너뜀")
            continue

        print(f">>> Size {i}/{total} - {pid}", flush=True)
        msgs = build_messages_choose_size(ptext)
        raw = generate_once(msgs, **GEN_ARGS)
        size_num = parse_size_num_strict(raw)
        size_label = size_num_to_label(size_num)

        results.append({
            "persona_id": pid,
            "persona_idx": row.get("persona_idx"),
            "variant_num_prev": row.get("variant_num"),
            "variant_choice_prev": row.get("variant_choice"),
            "size_num": size_num,          # 1/2
            "size_choice": size_label,     # 200g / 340g
            "raw_reply": raw,              # 원문(숫자 1글자)
        })

    # 새 CSV로 저장
    with open(out_path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(
            f,
            fieldnames=[
                "persona_id","persona_idx",
                "variant_num_prev","variant_choice_prev",
                "size_num","size_choice","raw_reply"
            ]
        )
        w.writeheader()
        w.writerows(results)

    print(f"\n✅ Saved size choices: {os.path.abspath(out_path)} (rows={len(results)})")

# 사용 예시
run_size_choice_for_omeletham_buyers(
    prev_csv_path="/content/dongwon_variant_choice_omeletham.csv",
    out_path="/content/dongwon_omeletham_size_choice.csv"
)

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd

IN_FILE = "dongwon_omeletham_size_choice.csv"
OUT_SUMMARY = "dongwon_omeletham_size_choice_summary.csv"

NUM2LABEL = {1: "200g", 2: "340g"}

def analyze_omeletham_size_choices(in_path: str = IN_FILE, out_path: str = OUT_SUMMARY):
    # 1) 데이터 로드
    df = pd.read_csv(in_path, encoding="utf-8-sig")

    # 2) 최소 필요 컬럼 확인
    required_cols = {"persona_id","persona_idx","size_num","size_choice","raw_reply"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"CSV에 필요한 컬럼이 없습니다: {missing}")

    # 3) 숫자/라벨 정합성 보정
    df["size_num"] = pd.to_numeric(df["size_num"], errors="coerce").fillna(2).astype(int)
    df["size_num"] = df["size_num"].where(df["size_num"].isin([1, 2]), 2)
    df["size_choice"] = df["size_num"].map(NUM2LABEL)

    # 4) 집계
    order = [1, 2]
    counts = df["size_num"].value_counts().reindex(order, fill_value=0).astype(int)
    total = int(counts.sum())

    if total > 0:
        ratios = (counts / total).round(6)       # 합=1
        percents = (ratios * 100).round(2)       # 0~100 %
    else:
        ratios = pd.Series([0.0, 0.0], index=order)
        percents = pd.Series([0.0, 0.0], index=order)

    # 5) 요약 테이블
    summary = pd.DataFrame({
        "size_num": order,
        "size_choice": [NUM2LABEL[n] for n in order],
        "count": counts.values,
        "percent": percents.values,              # 0~100
        "ratio": ratios.values                   # 합=1
    })

    # 6) 출력
    print("=== 오믈레햄 구매자: 용량 선택 요약 ===")
    print(f"총 대상 수: {total}명")
    for n in order:
        print(f"{n} = {NUM2LABEL[n]}: {counts[n]}명 ({percents[n]}%), ratio={ratios[n]:.3f}")

    print("\n=== 요약 테이블 ===")
    print(summary.to_string(index=False))

    # 7) 저장
    summary.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ 요약 저장: {out_path}")

    # 8) 반환(추가 분석용)
    return df, summary

if __name__ == "__main__":
    _, _ = analyze_omeletham_size_choices()


# 통조림 햄 시뮬레이션 최종 정리

리챔 오리지널 : 리챔 오믈레햄 = 93 : 7

리챔 오믈레햄 → 200g : 340g = 5 : 2 = 2.5 : 1

# 카페라떼/바닐라라떼 시뮬레이션

In [ ]:
import re, csv, os
from typing import List, Tuple

with open("Latte.txt", "r", encoding="utf-8") as f:
    PERSONAS_RAW = f.read()

print("\n".join(PERSONAS_RAW.splitlines()[:5]))

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple

# =========================
# 0) 전역: 상품/설명 (참고용, 균형 서술)
# =========================
DONGWON_VARIANTS = (
    "[동원 신제품 종류]\n"
    "- 카페라떼: 유당을 제거한 소화가 잘되는 우유를 사용, 신선한 1등급 원유와 크림을 더해 고소하면서 진한 라떼 본연의 맛을 그대로 담아 속이 편하면서 부드럽고 진한 카페라떼\n"
    "- 바닐라라떼: 유당을 제거한 소화가 잘되는 우유를 사용, 신선한 1등급 원유와 부드러운 크림을 더하고 은은한 바닐라향이 어우러져 입안 가득 부드럽고 풍부한 바닐라 라떼."
)

MARKET_INFO = (
    "[시장 정보]\n"
    "- 덴마크 우유(동원) 카페라떼: 250ml 2800원 수준.\n"
    "- 덴마크 우유(동원) 바닐라라떼: 250ml 2800원 수준.\n"
    "- 빙그레 카페라떼: 250ml 2000원 수준.\n"
    "- 빙그레 바닐라라떼: 250ml 2000원 수준.\n"
    "- 매일유업 카페라떼: 220ml 2100원 수준.\n"
    "- 매일유업 바닐라라떼: 325ml 3000원 수준.\n"
    "- 서울우유 카페라떼: 250ml 2600원 수준.\n"
    "- 서울우유 바닐라라떼: 250ml 2600원 수준.\n"
)

# =========================
# 1) 페르소나 파서
# =========================
def parse_personas(raw: str) -> List[Tuple[str, str]]:
    """
    PERSONAS_RAW 형식 예:
    Persona_001
    ...설명...
    Persona_002
    ...설명...
    """
    text = raw.strip().replace("\r\n", "\n").replace("\r", "\n")
    header_re = re.compile(r"^\s*\*{0,3}\s*Persona_(\d{3})\s*\*{0,3}\s*:?\s*$", re.M)
    matches = list(header_re.finditer(text))
    pairs: List[Tuple[str, str]] = []
    for i, m in enumerate(matches):
        pid = f"Persona_{m.group(1)}"
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        block = text[start:end].strip()
        if block:
            pairs.append((pid, block))
    pairs.sort(key=lambda x: int(x[0].split("_")[1]))
    return pairs

# 외부에서 PERSONAS_RAW 문자열을 주입해 주세요.
# 예) personas = parse_personas(PERSONAS_RAW)

# =========================
# 2) 프롬프트 템플릿 (숫자 1/2 한 글자 출력 강제)
# =========================
SYSTEM_INST = (
    "너는 시장 조사/소비자 행동 분석 어시스턴트다.\n"
    "규칙:\n"
    " - 아래 [ROLE]의 페르소나 입장에서 덴마크 우유(동원) 커피의 맛 중 하나를 선택한다.\n"
    " - 출력 형식: 반드시 숫자 '1' 또는 '2' 중 **한 글자만** 출력한다.\n"
    "   1 = 카페라떼\n"
    "   2 = 바닐라라떼\n"
    " - **숫자 외의 모든 문자, 공백, 개행, 마침표, 설명을 절대 출력하지 않는다.**"
)

def persona_to_role_block(ptext: str) -> str:
    return (
        "[ROLE]\n"
        "아래 페르소나 설명을 그대로 따르는 소비자 역할을 맡아 실제 생활 맥락에서 판단한다.\n"
        f"{ptext}\n"
    )

def build_messages_choose_variant(persona_text: str):
    role_block = persona_to_role_block(persona_text)
    SELECTION_GUIDE = (
    "[선택 기준]\n"
    "- 1(카페라떼): 커피·우유 본연의 고소하고 담백한 맛을 선호하고, 첨가 향/단맛을 최소화한 ‘기본 라떼’를 찾을 때 적합. 식사와 곁들이기, 아침 한 잔, 공부·업무 등 집중이 필요한 상황처럼 깔끔한 끝맛과 약간의 로스티함을 긍정적으로 받아들이는 경우 선택.\n"
    "- 2(바닐라라떼): 은은한 바닐라 향과 부드러운 단맛으로 쓴맛을 누그러뜨린 조화를 선호할 때 적합. 디저트·베이커리와의 페어링, **오후/저녁** 간식·휴식·기분전환용 한 잔처럼 ‘달콤한 만족감’을 원하는 상황에 잘 맞음. **가끔 라떼에 시럽을 추가해 마시거나**, 커피가 쓰게 느껴지는 편, **동반자/가족의 단맛 취향을 배려**하려는 경우에도 선택.\n"
    "(결정 보조: 섭취 목적이 ‘식사 동반·아침·집중/담백 선호’에 가깝다면 1, ‘간식·휴식·디저트/단맛·향미 선호(특히 오후/저녁)’에 가깝다면 2를 고른다. 편향 금지: 두 선택지 중 어느 한쪽을 기계적으로 선호/회피하지 말고 페르소나의 생활 맥락과 취향에 근거해 판단한다.)"
    )
    QUESTION = (
        "[질문]\n"
        "덴마크 우유(동원) 커피의 맛 타입 중에서 무엇을 고를 것인가?\n"
        "선택지는 정확히 다음 둘 중 하나다:\n"
        "  1 = 카페라떼\n"
        "  2 = 바닐라라떼\n"
        "최종 답변은 반드시 1, 2 중 하나의 숫자만 출력하라."
    )
    user_content = (
        f"{role_block}\n"
        f"{MARKET_INFO}\n\n"
        f"{DONGWON_VARIANTS}\n\n"
        f"{SELECTION_GUIDE}\n\n"
        f"{QUESTION}\n"
    )
    return [
        {"role": "system", "content": SYSTEM_INST},
        {"role": "user",   "content": user_content},
    ]

# =========================
# 3) LLM 호출/검증 (한 글자 산출, 결정론적)
# =========================
# 주의: 아래 pipe는 외부에서 준비된 transformers pipeline(or API) 객체라고 가정합니다.
# 예) pipe = pipeline("chat", model=..., tokenizer=..., device_map="cpu")
GEN_ARGS = dict(
    do_sample=False,
    temperature=0.0,
    top_p=1.0,
    return_full_text=False,
    max_new_tokens=1,   # 한 글자만 생성
)

def _extract_assistant_text_from_pipe_output(out) -> str:
    if not out:
        return ""
    item = out[0]
    if isinstance(item, dict):
        if isinstance(item.get("generated_text"), str):
            return item["generated_text"]
        gen = item.get("generated_text")
        if isinstance(gen, list):
            for m in gen:
                if isinstance(m, dict) and m.get("role") == "assistant":
                    c = m.get("content", "")
                    if isinstance(c, str):
                        return c
        if isinstance(item.get("content"), str):
            return item["content"]
    if isinstance(item, str):
        return item
    return str(item)

def generate_once(messages, **gen_args) -> str:
    out = pipe(messages, **gen_args)
    text = _extract_assistant_text_from_pipe_output(out)
    return text.strip() if text else ""

def parse_variant_num_strict(text: str) -> int:
    """
    보정 없이 엄격 검증: 반드시 '1'/'2' 중 하나여야 함.
    아니면 예외 발생.
    """
    t = (text or "").strip()
    if t not in ("1", "2"):
        raise ValueError(f"모델 출력이 1/2가 아님: {repr(t)}")
    return int(t)

def num_to_label(n: int) -> str:
    return {1: "카페라떼", 2: "바닐라라떼"}[n]

# =========================
# 4) 실행 & 저장
# =========================
def main(personas: List[Tuple[str, str]], out_path: str = "dongwon_variant_choice_latte.csv"):
    if not personas:
        raise ValueError("파싱된 페르소나가 없습니다. PERSONAS_RAW를 확인하세요.")

    rows = []
    total = len(personas)
    for i, (pid, ptext) in enumerate(personas, start=1):
        print(f">>> Persona {i}/{total} - {pid}", flush=True)
        msgs = build_messages_choose_variant(ptext)
        raw = generate_once(msgs, **GEN_ARGS)
        variant_num = parse_variant_num_strict(raw)  # 보정 없이 검증
        choice_label = num_to_label(variant_num)
        rows.append({
            "persona_idx": i,
            "persona_id": pid,
            "variant_num": variant_num,      # 1/2
            "variant_choice": choice_label,  # 카페라떼 / 바닐라라떼
            "raw_reply": raw,                # 원문(숫자 1글자)
        })

    with open(out_path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(
            f,
            fieldnames=["persona_idx","persona_id","variant_num","variant_choice","raw_reply"]
        )
        w.writeheader()
        w.writerows(rows)

    print(f"\n✅ Saved: {os.path.abspath(out_path)} (rows={len(rows)})")

# 사용 예시:
personas = parse_personas(PERSONAS_RAW)
if __name__ == "__main__":
    main(personas)


In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd

IN_FILE = "dongwon_variant_choice_latte.csv"
OUT_SUMMARY = "dongwon_variant_choice_latte_summary.csv"

NUM2LABEL = {1: "카페라떼", 2: "바닐라라떼"}

def analyze_latte_variant_choices(in_path: str = IN_FILE, out_path: str = OUT_SUMMARY):
    # 1) 데이터 로드
    df = pd.read_csv(in_path, encoding="utf-8-sig")

    # 2) 최소 필요 컬럼 확인
    required_cols = {"persona_idx","persona_id","variant_num","variant_choice","raw_reply"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"CSV에 필요한 컬럼이 없습니다: {missing}")

    # 3) 숫자/라벨 정합성 보정 (숫자 기준으로 라벨 재매핑)
    df["variant_num"] = pd.to_numeric(df["variant_num"], errors="coerce").fillna(2).astype(int)
    df["variant_num"] = df["variant_num"].where(df["variant_num"].isin([1, 2]), 2)
    df["variant_choice"] = df["variant_num"].map(NUM2LABEL)

    # 4) 집계
    order = [1, 2]
    counts = df["variant_num"].value_counts().reindex(order, fill_value=0).astype(int)
    total = int(counts.sum())

    if total > 0:
        ratios = (counts / total).round(6)    # 0~1, 합=1
        percents = (ratios * 100).round(2)    # 0~100 %
    else:
        ratios = pd.Series([0.0, 0.0], index=order)
        percents = pd.Series([0.0, 0.0], index=order)

    # 5) 요약 테이블
    summary = pd.DataFrame({
        "variant_num": order,
        "variant_choice": [NUM2LABEL[n] for n in order],
        "count": counts.values,
        "percent": percents.values,
        "ratio": ratios.values
    })

    # 6) 출력
    print("=== 동원 라떼: 맛 선택 요약 ===")
    print(f"총 응답 수: {total}명")
    for n in order:
        print(f"{n} = {NUM2LABEL[n]}: {counts[n]}명 ({percents[n]}%), ratio={ratios[n]:.3f}")

    print("\n=== 요약 테이블 ===")
    print(summary.to_string(index=False))

    # 7) 저장
    summary.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ 요약 저장: {out_path}")

    # 8) 반환(추가 분석용)
    return df, summary

if __name__ == "__main__":
    _, _ = analyze_latte_variant_choices()


# 카페라떼/바닐라라떼 시뮬레이션 최종 정리

카페라떼 : 바닐라라떼 = 72 : 28 = 2.6 : 1

# 참치액 시뮬레이션

In [ ]:
import re, csv, os
from typing import List, Tuple

with open("chamchiaek.txt", "r", encoding="utf-8") as f:
    PERSONAS_RAW = f.read()

print("\n".join(PERSONAS_RAW.splitlines()[:5]))

In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple

# =========================
# 0) 전역: 상품/설명 (참고용, 균형 서술)
# =========================
DONGWON_VARIANTS = (
    "[동원 신제품 종류]\n"
    "- 순한 맛: 훈연향을 낮추고 멸치 숙성액 기반의 깔끔한 감칠맛. 일상 메뉴 전반에 무난하게 어울림.\n"
    "- 진한 맛: 가쓰오 풍미가 또렷하고 존재감 있는 국물 맛을 내기 좋음. 전골·찌개·조림에 적합.\n"
    "- 프리미엄: 황다랑어 추출물 기반의 깊고 풍부한 감칠맛. 재료 맛을 살리며 특별식·접대상에도 어울림."
)

MARKET_INFO = (
    "[시장 정보]\n"
    "- 한라식품 참치액: 900ml 8,000원대.\n"
    "- 동원 참치액: 엑기스 함량 80% 이상, 900ml 6,500원 수준.\n"
    "- 사조 참치액: 900ml 6,300원 수준."
)

# =========================
# 1) 페르소나 파서
# =========================
def parse_personas(raw: str) -> List[Tuple[str, str]]:
    """
    PERSONAS_RAW 형식 예:
    Persona_001
    ...설명...
    Persona_002
    ...설명...
    """
    text = raw.strip().replace("\r\n", "\n").replace("\r", "\n")
    header_re = re.compile(r"^\s*\*{0,3}\s*Persona_(\d{3})\s*\*{0,3}\s*:?\s*$", re.M)
    matches = list(header_re.finditer(text))
    pairs: List[Tuple[str, str]] = []
    for i, m in enumerate(matches):
        pid = f"Persona_{m.group(1)}"
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        block = text[start:end].strip()
        if block:
            pairs.append((pid, block))
    pairs.sort(key=lambda x: int(x[0].split("_")[1]))
    return pairs

# =========================
# 2) 프롬프트 템플릿 (숫자 1/2/3 한 글자 출력 강제)
# =========================
SYSTEM_INST = (
    "너는 시장 조사/소비자 행동 분석 어시스턴트다.\n"
    "규칙:\n"
    " - 아래 [ROLE]의 페르소나 입장에서 동원 참치액의 맛을 하나 선택한다.\n"
    " - 출력 형식: 반드시 숫자 '1' 또는 '2' 또는 '3' 중 **한 글자만** 출력한다.\n"
    "  1 = 순한 맛\n"
    "  2 = 진한 맛\n"
    "  3 = 프리미엄\n"
    " - **숫자 외의 모든 문자, 공백, 개행, 마침표, 설명을 절대 출력하지 않는다.**"
)

def persona_to_role_block(ptext: str) -> str:
    return (
        "[ROLE]\n"
        "아래 페르소나 설명을 그대로 따르는 소비자 역할을 맡아 실제 생활 맥락에서 판단한다.\n"
        f"{ptext}\n"
    )

def build_messages_choose_variant(persona_text: str):
    role_block = persona_to_role_block(persona_text)
    SELECTION_GUIDE = (
        "[선택 기준]\n"
        "- 1(순한 맛): 가족 구성원 취향이 다양하거나, 일상 요리에 보편적으로 두루 쓰기에 적합함.\n"
        "- 2(진한 맛): 국물·전골에서 두드러지는 감칠맛과 향을 선호.\n"
        "- 3(프리미엄): 재료 본연의 맛을 살리며 깊은 여운을 느낄 수 있어 요리를 잘 하지 못해도 맛있는 상차림에 적합.\n"
        "(편향 금지: 세 선택지 중 어느 한쪽을 선호하거나 회피하지 말고, 페르소나 맥락에만 근거해 결정.)"
    )
    QUESTION = (
        "[질문]\n"
        "동원 참치액의 맛 타입 중에서 무엇을 고를 것인가?\n"
        "선택지는 정확히 다음 셋 중 하나다:\n"
        "  1 = 순한 맛\n"
        "  2 = 진한 맛\n"
        "  3 = 프리미엄\n"
        "최종 답변은 반드시 1, 2, 3 중 하나의 숫자만 출력하라."
    )
    user_content = (
        f"{role_block}\n"
        f"{MARKET_INFO}\n\n"
        f"{DONGWON_VARIANTS}\n\n"
        f"{SELECTION_GUIDE}\n\n"
        f"{QUESTION}\n"
    )
    return [
        {"role": "system", "content": SYSTEM_INST},
        {"role": "user",   "content": user_content},
    ]

# =========================
# 3) LLM 호출/검증 (한 글자 산출, 결정론적)
# =========================
GEN_ARGS = dict(
    do_sample=False,
    temperature=0.0,
    top_p=1.0,
    return_full_text=False,
    max_new_tokens=1,   # 한 글자만 생성
)

def _extract_assistant_text_from_pipe_output(out) -> str:
    if not out:
        return ""
    item = out[0]
    if isinstance(item, dict):
        if isinstance(item.get("generated_text"), str):
            return item["generated_text"]
        gen = item.get("generated_text")
        if isinstance(gen, list):
            for m in gen:
                if isinstance(m, dict) and m.get("role") == "assistant":
                    c = m.get("content", "")
                    if isinstance(c, str):
                        return c
        if isinstance(item.get("content"), str):
            return item["content"]
    if isinstance(item, str):
        return item
    return str(item)

def generate_once(messages, **gen_args) -> str:
    out = pipe(messages, **gen_args)
    text = _extract_assistant_text_from_pipe_output(out)
    return text.strip() if text else ""

def parse_variant_num_strict(text: str) -> int:
    t = (text or "").strip()
    if t not in ("1", "2", "3"):
        raise ValueError(f"모델 출력이 1/2/3이 아님: {repr(t)}")
    return int(t)

def num_to_label(n: int) -> str:
    return {1: "순한 맛", 2: "진한 맛", 3: "프리미엄"}[n]

# =========================
# 4) 실행 & 저장
# =========================
def main(personas: List[Tuple[str, str]], out_path: str = "dongwon_variant_choice_chamchiaek.csv"):
    if not personas:
        raise ValueError("파싱된 페르소나가 없습니다. PERSONAS_RAW를 확인하세요.")

    rows = []
    total = len(personas)
    for i, (pid, ptext) in enumerate(personas, start=1):
        print(f">>> Persona {i}/{total} - {pid}", flush=True)
        msgs = build_messages_choose_variant(ptext)
        raw = generate_once(msgs, **GEN_ARGS)
        variant_num = parse_variant_num_strict(raw)
        choice_label = num_to_label(variant_num)
        rows.append({
            "persona_idx": i,
            "persona_id": pid,
            "variant_num": variant_num,
            "variant_choice": choice_label,
            "raw_reply": raw,
        })

    with open(out_path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(
            f,
            fieldnames=["persona_idx","persona_id","variant_num","variant_choice","raw_reply"]
        )
        w.writeheader()
        w.writerows(rows)

    print(f"\n✅ Saved: {os.path.abspath(out_path)} (rows={len(rows)})")

# 사용 예시
personas = parse_personas(PERSONAS_RAW)
if __name__ == "__main__":
    main(personas)


In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd

IN_FILE = "dongwon_variant_choice_chamchiaek.csv"
OUT_SUMMARY = "dongwon_variant_choice_chamchiaek_summary.csv"

NUM2LABEL = {1: "순한 맛", 2: "진한 맛", 3: "프리미엄"}

def analyze_chamchiaek_variant_choices(in_path: str = IN_FILE, out_path: str = OUT_SUMMARY):
    # 1) 데이터 로드
    df = pd.read_csv(in_path, encoding="utf-8-sig")

    # 2) 최소 필요 컬럼 확인
    required_cols = {"persona_idx","persona_id","variant_num","variant_choice","raw_reply"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"CSV에 필요한 컬럼이 없습니다: {missing}")

    # 3) 숫자/라벨 정합성 보정 (숫자 기준으로 라벨 재매핑)
    df["variant_num"] = pd.to_numeric(df["variant_num"], errors="coerce").fillna(2).astype(int)
    df["variant_num"] = df["variant_num"].where(df["variant_num"].isin([1, 2, 3]), 2)
    df["variant_choice"] = df["variant_num"].map(NUM2LABEL)

    # 4) 집계
    order = [1, 2, 3]
    counts = df["variant_num"].value_counts().reindex(order, fill_value=0).astype(int)
    total = int(counts.sum())

    if total > 0:
        ratios = (counts / total).round(6)      # 0~1, 합=1
        percents = (ratios * 100).round(2)      # 0~100 %
    else:
        ratios = pd.Series([0.0, 0.0, 0.0], index=order)
        percents = pd.Series([0.0, 0.0, 0.0], index=order)

    # 5) 요약 테이블
    summary = pd.DataFrame({
        "variant_num": order,
        "variant_choice": [NUM2LABEL[n] for n in order],
        "count": counts.values,
        "percent": percents.values,             # 0~100
        "ratio": ratios.values                  # 합=1
    })

    # 6) 출력
    print("=== 동원 참치액: 맛 선택 요약 (3지선다) ===")
    print(f"총 응답 수: {total}명")
    for n in order:
        print(f"{n} = {NUM2LABEL[n]}: {counts[n]}명 ({percents[n]}%), ratio={ratios[n]:.3f}")

    print("\n=== 요약 테이블 ===")
    print(summary.to_string(index=False))

    # 7) 저장
    summary.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ 요약 저장: {out_path}")

    # 8) 반환(추가 분석용)
    return df, summary

if __name__ == "__main__":
    _, _ = analyze_chamchiaek_variant_choices()


# 참치액 순 시뮬레이션

In [ ]:
# =========================
# 순한 맛 구매자: 용량(500ml vs 900ml) 선택 & 저장
# =========================

# (1) 사이즈 선택용 시스템 프롬프트
SAUCE_SIZE_SYSTEM_INST = (
    "너는 시장 조사/소비자 행동 분석 어시스턴트다.\n"
    "규칙:\n"
    " - 아래 [ROLE]의 페르소나 입장에서 '동원 참치액(순한 맛)' 구매 시 용량 하나를 선택한다.\n"
    " - 출력 형식: 반드시 숫자 '1' 또는 '2' 중 **한 글자만** 출력한다.\n"
    "   1 = 500ml\n"
    "   2 = 900ml\n"
    " - **숫자 외의 모든 문자, 공백, 개행, 마침표, 설명을 절대 출력하지 않는다.**"
)

def build_messages_choose_sun_size(persona_text: str):
    role_block = persona_to_role_block(persona_text)
    SAUCE_SIZE_SELECTION_GUIDE = (
        "[선택 기준]\n"
        "- 1(500ml): 저렴함을 원하는 사람들이면 선택함. 국수, 찌개와 나물 등 다양한 요리를 시도하고 활용하기에 최적이다. 개봉 후 **신선도**  부담을 줄일 수 있어. 주로 냉장보관 공간이 작은 환경에 거주하는 청년층과 1~2인 가구에게도 적합하다. \n"
        "- 2(900ml): 요리를 주 4~5회 이상 자주 하며, 주로 비슷한 메뉴를 대량으로 여러번 먹을 때 산다. 대용량 냉장보관 공간이 충분하다면 적합하다.그러나 장기간 개봉 시 산폐 우려가 있어 이런 경우에는 1이 적절하다.\n"
        "편향 금지: 두 선택지 중 어느 한쪽을 기계적으로 선호/회피하지 말고 페르소나의 생활 맥락과 취향에 근거해 판단한다."
    )
    SAUCE_QUESTION_SIZE = (
        "[질문]\n"
        "동원 참치액(순한 맛)을 구매할 때 어떤 용량을 고를 것인가?\n"
        "선택지는 정확히 다음 둘 중 하나다:\n"
        "  1 = 500ml\n"
        "  2 = 900ml\n"
        "최종 답변은 반드시 1, 2 중 하나의 숫자만 출력하라."
    )
    user_content = (
        f"{role_block}\n"
        f"{MARKET_INFO}\n\n"        # 가격/시장 정보 재사용
        f"{DONGWON_VARIANTS}\n\n"   # 맛 설명 재사용
        f"{SAUCE_SIZE_SELECTION_GUIDE}\n\n"
        f"{SAUCE_QUESTION_SIZE}\n"
    )
    return [
        {"role": "system", "content": SAUCE_SIZE_SYSTEM_INST},
        {"role": "user",   "content": user_content},
    ]

# (2) '1/2' 엄격 파서 (사이즈용)
def parse_sauce_size_num_strict(text: str) -> int:
    t = (text or "").strip()
    if t not in ("1", "2"):
        raise ValueError(f"모델 출력이 1/2가 아님: {repr(t)}")
    return int(t)

def sauce_size_num_to_label(n: int) -> str:
    return {1: "500ml", 2: "900ml"}[n]

# (3) 실행 함수: 이전 CSV에서 '순한 맛(1)' 구매자만 추출해 용량 선택 저장
def run_size_choice_for_sun_sauce_buyers(
    prev_csv_path: str = "dongwon_variant_choice.csv",
    out_path: str = "tuna_sauce_sun_size_choice.csv"
):
    # 페르소나 맵 (id -> 텍스트)
    personas_list = parse_personas(PERSONAS_RAW)
    pid_to_text = {pid: ptext for pid, ptext in personas_list}

    targets = []
    with open(prev_csv_path, encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            # '순한 맛(1)'만 대상
            if row.get("variant_num") == "1" or row.get("variant_choice") == "순한 맛":
                targets.append(row)

    if not targets:
        print("대상(순한 맛 선택자)이 없습니다.")
        return

    results = []
    total = len(targets)
    for i, row in enumerate(targets, start=1):
        pid = row.get("persona_id")
        ptext = pid_to_text.get(pid, "")
        if not ptext:
            print(f"경고: 페르소나 텍스트 누락 - {pid}, 건너뜀")
            continue

        print(f">>> Sauce Size {i}/{total} - {pid}", flush=True)
        msgs = build_messages_choose_sun_size(ptext)
        raw = generate_once(msgs, **GEN_ARGS)
        size_num = parse_sauce_size_num_strict(raw)
        size_label = sauce_size_num_to_label(size_num)

        results.append({
            "persona_id": pid,
            "persona_idx": row.get("persona_idx"),
            "variant_num_prev": row.get("variant_num"),
            "variant_choice_prev": row.get("variant_choice"),
            "size_num": size_num,          # 1/2
            "size_choice": size_label,     # 500ml / 900ml
            "raw_reply": raw,              # 원문(숫자 1글자)
        })

    # 새 CSV로 저장 (순한 맛 선택자만)
    with open(out_path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(
            f,
            fieldnames=[
                "persona_id","persona_idx",
                "variant_num_prev","variant_choice_prev",
                "size_num","size_choice","raw_reply"
            ]
        )
        w.writeheader()
        w.writerows(results)

    print(f"\n✅ Saved sauce size choices: {os.path.abspath(out_path)} (rows={len(results)})")

# 사용 예시:
run_size_choice_for_sun_sauce_buyers(
    prev_csv_path="/content/dongwon_variant_choice_chamchiaek.csv",
    out_path="/content/tuna_sauce_sun_size_choice.csv"
    )

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd

IN_FILE = "tuna_sauce_sun_size_choice.csv"
OUT_SUMMARY = "tuna_sauce_sun_size_choice_summary.csv"

NUM2LABEL = {1: "500ml", 2: "900ml"}

def analyze_sun_sauce_size_choices(in_path: str = IN_FILE, out_path: str = OUT_SUMMARY):
    # 1) 데이터 로드
    df = pd.read_csv(in_path, encoding="utf-8-sig")

    # 2) 필요한 컬럼 확인
    required_cols = {"persona_id","persona_idx","size_num","size_choice","raw_reply"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"CSV에 필요한 컬럼이 없습니다: {missing}")

    # 3) 숫자/라벨 정합성 보정
    df["size_num"] = pd.to_numeric(df["size_num"], errors="coerce").fillna(2).astype(int)
    df["size_num"] = df["size_num"].where(df["size_num"].isin([1, 2]), 2)
    df["size_choice"] = df["size_num"].map(NUM2LABEL)

    # 4) 집계
    order = [1, 2]
    counts = df["size_num"].value_counts().reindex(order, fill_value=0).astype(int)
    total = int(counts.sum())

    if total > 0:
        ratios = (counts / total).round(6)     # 0~1, 합=1
        percents = (ratios * 100).round(2)     # 0~100 %
    else:
        ratios = pd.Series([0.0, 0.0], index=order)
        percents = pd.Series([0.0, 0.0], index=order)

    # 5) 요약 테이블
    summary = pd.DataFrame({
        "size_num": order,
        "size_choice": [NUM2LABEL[n] for n in order],
        "count": counts.values,
        "percent": percents.values,            # 0~100
        "ratio": ratios.values                 # 합=1
    })

    # 6) 출력
    print("=== 순한 맛 구매자: 용량 선택 요약 ===")
    print(f"총 대상 수: {total}명")
    for n in order:
        print(f"{n} = {NUM2LABEL[n]}: {counts[n]}명 ({percents[n]}%), ratio={ratios[n]:.3f}")

    print("\n=== 요약 테이블 ===")
    print(summary.to_string(index=False))

    # 7) 저장
    summary.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ 요약 저장: {out_path}")

    # 8) 반환(추가 분석용)
    return df, summary

if __name__ == "__main__":
    _, _ = analyze_sun_sauce_size_choices()


# 참치액 진 시뮬레이션

In [ ]:
# =========================
# 진한 맛 구매자: 용량(500ml vs 900ml) 선택 & 저장
# =========================

# (1) 사이즈 선택용 시스템 프롬프트
JIN_SIZE_SYSTEM_INST = (
    "너는 시장 조사/소비자 행동 분석 어시스턴트다.\n"
    "규칙:\n"
    " - 아래 [ROLE]의 페르소나 입장에서 '동원 참치액(진한 맛)' 구매 시 용량 하나를 선택한다.\n"
    " - 출력 형식: 반드시 숫자 '1' 또는 '2' 중 **한 글자만** 출력한다.\n"
    "   1 = 500ml\n"
    "   2 = 900ml\n"
    " - **숫자 외의 모든 문자, 공백, 개행, 마침표, 설명을 절대 출력하지 않는다.**"
)

def build_messages_choose_jin_size(persona_text: str):
    role_block = persona_to_role_block(persona_text)
    JIN_SAUCE_SIZE_SELECTION_GUIDE = (
   "- 1(500ml): 기본적으로 가장 무난한 선택이다. **1~2인 가구**뿐 아니라 **3인 가구라도 매일 사용하지 않는 경우** 적절하다. 다양한 요리에 조금씩 널리 쓰기 좋고, 신선도 문제에서 자유롭고 보관 부담이 적다. **외식·배달이 잦거나 사용량이 일정치 않을 때**에도 효과적이다.\n"
    "- 2(900ml): 특별히 **4인 이상 대가족**, 혹은 **3인 가구라도 매일 집밥을 하고 대량 조리를 자주 하는 경우**에만 맞다. 냉장·냉동 공간이 넉넉하고, 같은 메뉴를 반복해 먹을 계획이 확실해야 한다. 개봉 후 장기간 보관 시 산폐 위험이 있다. 이런 조건이 불확실하다면 1이 더 자연스럽다.\n"

    "편향 금지: 반드시 페르소나의 생활 패턴과 식습관에 따라 판단한다."
)
    JIN_SAUCE_QUESTION_SIZE = (
        "[질문]\n"
        "동원 참치액(진한 맛)을 구매할 때 어떤 용량을 고를 것인가?\n"
        "선택지는 정확히 다음 둘 중 하나다:\n"
        "  1 = 500ml\n"
        "  2 = 900ml\n"
        "최종 답변은 반드시 1, 2 중 하나의 숫자만 출력하라."
    )
    user_content = (
        f"{role_block}\n"
        f"{MARKET_INFO}\n\n"
        f"{DONGWON_VARIANTS}\n\n"
        f"{JIN_SAUCE_SIZE_SELECTION_GUIDE}\n\n"
        f"{JIN_SAUCE_QUESTION_SIZE}\n"
    )
    return [
        {"role": "system", "content": JIN_SIZE_SYSTEM_INST},
        {"role": "user",   "content": user_content},
    ]

# (2) '1/2' 엄격 파서 (사이즈용)
def parse_jin_size_num_strict(text: str) -> int:
    t = (text or "").strip()
    if t not in ("1", "2"):
        raise ValueError(f"모델 출력이 1/2가 아님: {repr(t)}")
    return int(t)

def jin_size_num_to_label(n: int) -> str:
    return {1: "500ml", 2: "900ml"}[n]

# (3) 실행 함수: 이전 CSV에서 '진한 맛(2)' 구매자만 추출해 용량 선택 저장
def run_size_choice_for_jin_sauce_buyers(
    prev_csv_path: str = "dongwon_variant_choice.csv",
    out_path: str = "tuna_sauce_jin_size_choice.csv"
):
    # 페르소나 맵 (id -> 텍스트)
    personas_list = parse_personas(PERSONAS_RAW)
    pid_to_text = {pid: ptext for pid, ptext in personas_list}

    targets = []
    with open(prev_csv_path, encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            # '진한 맛(2)'만 대상
            if row.get("variant_num") == "2" or row.get("variant_choice") == "진한 맛":
                targets.append(row)

    if not targets:
        print("대상(진한 맛 선택자)이 없습니다.")
        return

    results = []
    total = len(targets)
    for i, row in enumerate(targets, start=1):
        pid = row.get("persona_id")
        ptext = pid_to_text.get(pid, "")
        if not ptext:
            print(f"경고: 페르소나 텍스트 누락 - {pid}, 건너뜀")
            continue

        print(f">>> Jin Sauce Size {i}/{total} - {pid}", flush=True)
        msgs = build_messages_choose_jin_size(ptext)
        raw = generate_once(msgs, **GEN_ARGS)
        size_num = parse_jin_size_num_strict(raw)
        size_label = jin_size_num_to_label(size_num)

        results.append({
            "persona_id": pid,
            "persona_idx": row.get("persona_idx"),
            "variant_num_prev": row.get("variant_num"),
            "variant_choice_prev": row.get("variant_choice"),
            "size_num": size_num,          # 1/2
            "size_choice": size_label,     # 500ml / 900ml
            "raw_reply": raw,              # 원문(숫자 1글자)
        })

    # 새 CSV로 저장 (진한 맛 선택자만)
    with open(out_path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(
            f,
            fieldnames=[
                "persona_id","persona_idx",
                "variant_num_prev","variant_choice_prev",
                "size_num","size_choice","raw_reply"
            ]
        )
        w.writeheader()
        w.writerows(results)

    print(f"\n✅ Saved JIN sauce size choices: {os.path.abspath(out_path)} (rows={len(results)})")

# 사용 예시:
run_size_choice_for_jin_sauce_buyers(
    prev_csv_path="/content/dongwon_variant_choice_chamchiaek.csv",
    out_path="/content/tuna_sauce_jin_size_choice.csv"
    )

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd

IN_FILE = "/content/tuna_sauce_jin_size_choice.csv"
OUT_SUMMARY = "/content/tuna_sauce_jin_size_choice_summary.csv"

NUM2LABEL = {1: "500ml", 2: "900ml"}

def analyze_jin_sauce_size_choices(in_path: str = IN_FILE, out_path: str = OUT_SUMMARY):
    # 1) 데이터 로드
    df = pd.read_csv(in_path, encoding="utf-8-sig")

    # 2) 필요한 컬럼 확인
    required_cols = {"persona_id","persona_idx","size_num","size_choice","raw_reply"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"CSV에 필요한 컬럼이 없습니다: {missing}")

    # 3) 숫자/라벨 정합성 보정
    df["size_num"] = pd.to_numeric(df["size_num"], errors="coerce").fillna(1).astype(int)
    df["size_num"] = df["size_num"].where(df["size_num"].isin([1, 2]), 1)
    df["size_choice"] = df["size_num"].map(NUM2LABEL)

    # 4) 집계
    order = [1, 2]
    counts = df["size_num"].value_counts().reindex(order, fill_value=0).astype(int)
    total = int(counts.sum())

    if total > 0:
        ratios = (counts / total).round(6)      # 0~1, 합=1
        percents = (ratios * 100).round(2)      # 0~100 %
    else:
        # 방어 처리
        ratios = pd.Series([0.0, 0.0], index=order)
        percents = pd.Series([0.0, 0.0], index=order)

    # 5) 요약 테이블
    summary = pd.DataFrame({
        "size_num": order,
        "size_choice": [NUM2LABEL[n] for n in order],
        "count": counts.values,
        "percent": percents.values,             # 0~100
        "ratio": ratios.values                  # 합=1
    })

    # 6) 콘솔 출력
    print("=== 진한 맛 구매자: 용량 선택 요약 ===")
    print(f"총 대상 수: {total}명")
    for n in order:
        print(f"{n} = {NUM2LABEL[n]}: {counts[n]}명 ({percents[n]}%), ratio={ratios[n]:.3f}")

    print("\n=== 요약 테이블 ===")
    print(summary.to_string(index=False))

    # 7) 저장
    summary.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ 요약 저장: {out_path}")

    # 8) 반환(추가 분석용)
    return df, summary

if __name__ == "__main__":
    _, _ = analyze_jin_sauce_size_choices()


# 참치액 프리미엄 시뮬레이션

In [ ]:
import re, csv, os
from typing import List, Tuple

with open("chamchiaek.txt", "r", encoding="utf-8") as f:
    PERSONAS_RAW = f.read()

print("\n".join(PERSONAS_RAW.splitlines()[:5]))

In [ ]:
import pandas as pd
import re

# 파일 경로
csv_path = "/content/dongwon_variant_choice_chamchiaek.csv"
txt_path = "/content/chamchiaek.txt"

# 1) variant_choice CSV 불러오기
df = pd.read_csv(csv_path)

# variant_choice가 '프리미엄'인 persona만 추출
premium_ids = df.loc[df["variant_choice"]=="프리미엄", "persona_id"].tolist()

# 2) TXT 불러오기
with open(txt_path, "r", encoding="utf-8") as f:
    personas_raw = f.read()

# 3) Persona 블록 단위로 분리
blocks = [b.strip() for b in re.split(r"\n\s*\n", personas_raw) if b.strip()]

# 4) premium persona만 필터링
premium_blocks = []
for b in blocks:
    m = re.match(r"\*\*(Persona_\d+)\*\*", b)
    if m:
        pid = m.group(1)
        if pid in premium_ids:
            premium_blocks.append(b)

# 5) update된 persona_raws (프리미엄만 포함)
persona_raws_updated = "\n\n".join(premium_blocks)

personas_raw = persona_raws_updated


In [ ]:
# -*- coding: utf-8 -*-
import re, csv, os
from typing import List, Tuple, Dict, Optional
import pandas as pd

# ---------------------------------------------------------
# 0) 환경 폴백
# ---------------------------------------------------------
try:
    MARKET_INFO
except NameError:
    MARKET_INFO = ""

try:
    DONGWON_VARIANTS
except NameError:
    DONGWON_VARIANTS = (
        "[동원 신제품 종류]\n"
        "- 프리미엄: 황다랑어 추출물 기반의 깊고 풍부한 감칠맛. 재료 맛을 살리며 특별식·접대상에도 어울림."
    )

# 파이프라인 호출 래퍼 (결정적 추론)
def generate_once(messages, max_new_tokens=4, do_sample=False, return_full_text=False):
    """
    전역 pipeline 'pipe'가 이미 준비되어 있다고 가정.
    messages: [{"role":"system","content":"..."}, {"role":"user","content":"..."}]
    """
    out = pipe(messages, max_new_tokens=max_new_tokens, do_sample=do_sample, return_full_text=return_full_text)
    text = out[0].get("generated_text", out[0].get("text", ""))
    return (text or "").strip()

# ---------------------------------------------------------
# 1) 페르소나 파싱 유틸 — (persona_id, block_text) 리스트 반환
# ---------------------------------------------------------
def parse_personas_from_raw(text: str) -> List[Tuple[str, str]]:
    blocks = [b.strip() for b in re.split(r"\n\s*\n", text) if b.strip()]
    out: List[Tuple[str, str]] = []
    for b in blocks:
        m = re.match(r"\*\*(Persona_\d+)\*\*", b)
        if m:
            out.append((m.group(1), b))
    return out

def persona_to_role_block(persona_text: str, persona_id: Optional[str] = None) -> str:
    pid_line = f"persona_id: {persona_id}\n" if persona_id else ""
    return f"[ROLE]\n{pid_line}{persona_text}"

# ---------------------------------------------------------
# 2) 프리미엄 500g vs 900g 선택 프롬프트
# ---------------------------------------------------------
PREMIUM_SIZE_SYSTEM_INST = (
    "너는 시장 조사/소비자 행동 분석 어시스턴트다.\n"
    "규칙:\n"
    " - 아래 [ROLE]의 페르소나 입장에서 '동원 참치액(프리미엄)' 구매 시 용량 하나를 **신중히** 선택한다.\n"
    " - 출력 형식: 반드시 숫자 '1' 또는 '2' 중 **한 글자만** 출력한다.\n"
    "   1 = 500g\n"
    "   2 = 900g\n"
    " - **숫자 외의 모든 문자, 공백, 개행, 마침표, 설명을 절대 출력하지 않는다.**\n"
    "\n"
    "[편향 방지]\n"
    " - 성별/소득/브랜드 **단독 신호만으로** 결정하지 말고, 조리·소비 맥락을 우선한다.\n"
    " - 최근성·순서·확증 편향을 배제하고, 동일 조건에는 동일 판단을 내린다.\n"
    "\n"
    "[판단 기준]\n"
    " - 1~3인 가구, 사용 빈도 **주 0~4회**, 메뉴가 **자주 바뀌거나** 국물 외(볶음·무침·비빔 등)로 **소량 분산 사용**, **다음 끼니로 이어 쓰는 계획이 불명확**, **신선도/염분 관리** 중시, **공용/협소 냉장고·이동/도시락/오피스** 환경, **첫 구매/테스트** 상황이면 1이 합리적이다.\n"
    " - **2(900g)는 아래 ①~⑦을 주로 만족할 때만** 고려한다:\n"
    "   ① 가구 규모 **4인 이상**(또는 3인이라도 **주 6~7회** 지속 사용),\n"
    "   ② **전골·찌개·국물 중심의 대량 조리** 비중이 매우 높음,\n"
    "   ③ **48~72시간 내 연속 소진** 계획이 **확정**(소분/냉장·냉동 전제),\n"
    "   ④ **동일 메뉴를 며칠 연속** 먹는 데 거부감이 없음,\n"
    "   ⑤ **냉장/냉동 공간이 넉넉**하여 대용량 보관에 무리가 없음,\n"
    "   ⑥ **그람당 단가(가성비)를 최우선**으로 고려,\n"
    "   ⑦ 외식·배달 빈도 **주 0~1회** 수준으로 가정 내 조리 비중이 매우 높음.\n"
    " - 위 요건 중 **하나라도 불확실/미충족이면 1(500g)**으로 판단한다. 경계/동률은 1을 우선한다.\n"
)


PREMIUM_SIZE_SELECTION_GUIDE = (
    "[질문]\n"
    "동원 참치액(프리미엄)을 구매할 때 어떤 용량을 고를 것인가?\n"
    "선택지는 정확히 다음 둘 중 하나다:\n"
    "  1 = 500g\n"
    "  2 = 900g\n"
    "최종 답변은 반드시 1, 2 중 하나의 숫자만 출력하라."
)

def build_messages_choose_premium_size(persona_text: str, persona_id: Optional[str] = None):
    role_block = persona_to_role_block(persona_text, persona_id=persona_id)
    user_content = (
        f"{role_block}\n"
        f"{MARKET_INFO}\n\n"
        f"{DONGWON_VARIANTS}\n\n"
        f"{PREMIUM_SIZE_SELECTION_GUIDE}\n"
    )
    return [
        {"role": "system", "content": PREMIUM_SIZE_SYSTEM_INST},
        {"role": "user",   "content": user_content},
    ]

# ---------------------------------------------------------
# 3) 파서/라벨
# ---------------------------------------------------------
def parse_size_num_strict(text: str) -> int:
    t = (text or "").strip()
    if t not in ("1", "2"):
        raise ValueError(f"모델 출력이 1/2가 아님: {repr(t)}")
    return int(t)

def size_num_to_label(n: int) -> str:
    return {1: "500g", 2: "900g"}[n]

# ---------------------------------------------------------
# 4) 시뮬레이션 실행
# ---------------------------------------------------------
def run_premium_size_simulation_from_text(
    personas_text: str,
    out_csv_path: str = "/content/tuna_premium_size_choice.csv",
    max_new_tokens: int = 4
):
    # 1) 페르소나 파싱
    personas = parse_personas_from_raw(personas_text)
    if not personas:
        print("페르소나 텍스트가 비어있거나 파싱할 페르소나가 없습니다.")
        return

    # 2) 실행
    rows = []
    for i, (pid, ptext) in enumerate(personas, start=1):
        print(f">>> Premium Size {i}/{len(personas)} - {pid}")
        msgs = build_messages_choose_premium_size(ptext, persona_id=pid)
        raw = generate_once(msgs, max_new_tokens=max_new_tokens, do_sample=False, return_full_text=False)
        try:
            size_num = parse_size_num_strict(raw)
        except Exception as e:
            # 안전장치: 실패 시 기본 500g으로 보정
            print(f"경고: 잘못된 응답({repr(raw)}) → 500g으로 보정")
            size_num = 1
        size_label = size_num_to_label(size_num)

        rows.append({
            "persona_id": pid,
            "size_num": size_num,       # 1/2
            "size_choice": size_label,  # 500g/900g
            "raw_reply": raw,
        })

    # 3) 저장
    df_out = pd.DataFrame(rows, columns=["persona_id","size_num","size_choice","raw_reply"])
    os.makedirs(os.path.dirname(out_csv_path), exist_ok=True)
    df_out.to_csv(out_csv_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ Saved: {os.path.abspath(out_csv_path)} (rows={len(df_out)})")
    return df_out

# ---------------------------------------------------------
# 5) 실행
# ---------------------------------------------------------
df_result = run_premium_size_simulation_from_text(
    personas_text=personas_raw,
    out_csv_path="/content/tuna_premium_size_choice.csv",
    max_new_tokens=4
)

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd

IN_FILE = "/content/tuna_premium_size_choice.csv"
OUT_SUMMARY = "/content/tuna_premium_size_choice_summary.csv"

NUM2LABEL = {1: "500g", 2: "900g"}

def analyze_premium_size_choices(in_path: str = IN_FILE, out_path: str = OUT_SUMMARY):
    # 1) 데이터 로드
    df = pd.read_csv(in_path, encoding="utf-8-sig")

    # 2) 필요한 컬럼 확인
    required_cols = {"persona_id","size_num","size_choice","raw_reply"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"CSV에 필요한 컬럼이 없습니다: {missing}")

    # 3) 숫자/라벨 정합성 보정
    df["size_num"] = pd.to_numeric(df["size_num"], errors="coerce").fillna(1).astype(int)
    df["size_num"] = df["size_num"].where(df["size_num"].isin([1, 2]), 1)
    df["size_choice"] = df["size_num"].map(NUM2LABEL)

    # 4) 집계
    order = [1, 2]
    counts = df["size_num"].value_counts().reindex(order, fill_value=0).astype(int)
    total = int(counts.sum())

    if total > 0:
        ratios = (counts / total).round(6)     # 0~1, 합=1
        percents = (ratios * 100).round(2)     # 0~100 %
    else:
        ratios = pd.Series([0.0, 0.0], index=order)
        percents = pd.Series([0.0, 0.0], index=order)

    # 5) 요약 테이블
    summary = pd.DataFrame({
        "size_num": order,
        "size_choice": [NUM2LABEL[n] for n in order],
        "count": counts.values,
        "percent": percents.values,            # 0~100
        "ratio": ratios.values                 # 합=1
    })

    # 6) 출력
    print("=== 프리미엄 구매자: 용량 선택 요약 ===")
    print(f"총 대상 수: {total}명")
    for n in order:
        print(f"{n} = {NUM2LABEL[n]}: {counts[n]}명 ({percents[n]}%), ratio={ratios[n]:.3f}")

    print("\n=== 요약 테이블 ===")
    print(summary.to_string(index=False))

    # 7) 저장
    summary.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ 요약 저장: {out_path}")

    # 8) 반환(추가 분석용)
    return df, summary

if __name__ == "__main__":
    _, _ = analyze_premium_size_choices()


# 참치액 시뮬레이션 최종 정리

순 : 진 : 프리미엄 = 37 : 56 : 7

순 → 500ml : 900ml = 28 : 9 = 3.1 : 1

진 → 500ml : 900ml = 34 : 22 = 1.5 : 1

프리미엄 → 500ml : 900ml = 6 : 1